<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

March 2026 page-month feature set (331,437 pages, 55 clients):

| Field | Median | Mean | 90th %ile | 99th %ile | Max |
|---|---|---|---|---|---|
| impressions | 2 | 846.8 | 1,707 | 14,909 | 617,124 |
| clicks | 0 | 2.5 | 3 | 47 | 5,668 |
| ctr | 0.0% | 0.24% | 0.33% | 2.63% | 100% |
| avg_position | 2.41 | 8.53 | 26.42 | 73.05 | 309 |

**Heavy tail, confirmed concretely:** the top 1% of pages by impressions
hold 34.9% of all March traffic. The gap between median impressions (2) and
mean (846.8) — over 400x — says the same thing another way: most pages get
almost no traffic, and a small number of pages carry the volume.

**Implication for signal testing:** any signal test run on raw means will
be dominated by a handful of extreme pages. Section 2's tests use medians,
bucketed comparisons, or an explicit volume floor rather than raw averages,
to avoid a false "confirmed" or "false" verdict driven by outliers.

**A distribution oddity worth flagging, not hiding:** `avg_position` has a
max of 309 — well outside the ~1–100 range a normal SERP position report
would show. Worth noting as a caveat on any position-based signal test
below, since a handful of very high position values could skew a mean
(mitigated by using median/bucketed comparisons, same as above).

In [4]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

features = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr,
        AVG(gsc_avg_position) AS avg_position
    FROM {DAILY}
    GROUP BY content_hash_id, client_hash_id
""").df()
features["ctr"] = features["ctr"].fillna(0)
features["avg_position"] = features["avg_position"].fillna(0)

for col in ["impressions", "clicks", "ctr", "avg_position"]:
    print(f"--- {col} ---")
    print(features[col].describe(percentiles=[0.5, 0.9, 0.99]).to_string())
    print()

# Concrete heavy-tail check: what share of total impressions comes from the
# top 1% of pages?
top1pct_share = features.nlargest(int(len(features)*0.01), "impressions")["impressions"].sum() / features["impressions"].sum()
print(f"Share of total March impressions held by the top 1% of pages: {round(100*top1pct_share, 1)}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- impressions ---
count    331437.000000
mean        846.790156
std        4044.514753
min           0.000000
50%           2.000000
90%        1707.000000
99%       14909.280000
max      617124.000000

--- clicks ---
count    331437.000000
mean          2.479602
std          19.651282
min           0.000000
50%           0.000000
90%           3.000000
99%          47.000000
max        5668.000000

--- ctr ---
count    331437.000000
mean          0.244964
std           2.766870
min           0.000000
50%           0.000000
90%           0.330000
99%           2.626400
max         100.000000

--- avg_position ---
count    331437.000000
mean          8.531577
std          15.182651
min           0.000000
50%           2.412482
90%          26.420838
99%          73.045451
max         309.000000

Share of total March impressions held by the top 1% of pages: 34.9%


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

All tests use a 500-impression volume floor (61,924 pages, 18.7% of the
catalog) to avoid the long tail from Section 1 dominating the comparison.

**Signal #1 — CTR vs. position bucket: CONFIRMED.**
Median CTR falls cleanly and monotonically as position worsens: 0.26%
(pos 1-3) → 0.22% → 0.16% → 0.08% → 0.00% (pos 50+). This is the
strongest, cleanest signal of the three.

**Signal #2 — Impression volume vs. position bucket: MIXED.**
Median impressions mostly falls with worse position (2,553 → 2,013 →
1,361) but breaks pattern at the 21-50 bucket (2,103 — higher than 11-20's
1,361) before dropping again at 50+ (824). Not a clean monotonic
relationship — a real, disclosed exception, not smoothed over. Possible
explanation: pages ranking in the 21-50 range may include a mix of very
different query types (broad, high-volume terms a site ranks moderately
for, alongside narrow terms it ranks well for) — worth investigating
further, not asserting as fact here.

**Signal #3 — CTR vs. impression volume tier: CONFIRMED.**
Median CTR rises monotonically with volume quartile: 0.14% (lowest) →
0.17% → 0.19% → 0.22% (highest), even after controlling for the volume
floor. This means volume and CTR aren't independent — a real confound
worth keeping in mind when reading Signal 1 in isolation.

In [5]:
MIN_IMPRESSIONS = 500
eligible = features[features["impressions"] >= MIN_IMPRESSIONS].copy()
print(f"Pages meeting the {MIN_IMPRESSIONS}-impression floor: {len(eligible)} ({100*len(eligible)/len(features):.1f}%)")
print()

eligible["position_bucket"] = pd.cut(
    eligible["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

# --- Signal 1: CTR vs position bucket ---
sig1 = eligible.groupby("position_bucket", observed=True)["ctr"].median()
print("--- Signal 1: median CTR by position bucket ---")
print(sig1)
verdict1 = "CONFIRMED" if sig1.is_monotonic_decreasing else "MIXED"
print(f"Verdict: {verdict1} (CTR should decrease as position bucket worsens for CONFIRMED)")
print()

# --- Signal 2: impressions vs position bucket ---
sig2 = eligible.groupby("position_bucket", observed=True)["impressions"].median()
print("--- Signal 2: median impressions by position bucket ---")
print(sig2)
verdict2 = "CONFIRMED" if sig2.is_monotonic_decreasing else "MIXED"
print(f"Verdict: {verdict2} (impressions should decrease as position bucket worsens for CONFIRMED)")
print()

# --- Signal 3: CTR vs impression volume tier (within eligible pages only) ---
eligible["volume_tier"] = pd.qcut(eligible["impressions"], q=4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])
sig3 = eligible.groupby("volume_tier", observed=True)["ctr"].median()
print("--- Signal 3: median CTR by impression volume quartile ---")
print(sig3)
verdict3 = "MIXED" if not (sig3.is_monotonic_increasing or sig3.is_monotonic_decreasing) else ("CONFIRMED" if sig3.is_monotonic_increasing else "OPPOSITE")
print(f"Verdict: {verdict3}")

Pages meeting the 500-impression floor: 61924 (18.7%)

--- Signal 1: median CTR by position bucket ---
position_bucket
1-3      0.26
4-10     0.22
11-20    0.16
21-50    0.08
50+      0.00
Name: ctr, dtype: float64
Verdict: CONFIRMED (CTR should decrease as position bucket worsens for CONFIRMED)

--- Signal 2: median impressions by position bucket ---
position_bucket
1-3      2553.0
4-10     2013.0
11-20    1361.0
21-50    2103.0
50+       824.0
Name: impressions, dtype: float64
Verdict: MIXED (impressions should decrease as position bucket worsens for CONFIRMED)

--- Signal 3: median CTR by impression volume quartile ---
volume_tier
Q1 (lowest)     0.14
Q2              0.17
Q3              0.19
Q4 (highest)    0.22
Name: ctr, dtype: float64
Verdict: CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The Week 4 baseline rule (`low_ctr_visible_page` → `review_ctr_fix`) assumes
a 0.5% CTR benchmark meaningfully separates underperforming visible pages
from normal ones. Tested against 50,764 visible, eligible pages
(position ≤ 20, impressions ≥ 500).

**Result: the benchmark is miscalibrated — flags 80.8% of visible pages,**
far outside a reasonable "genuine minority" range (roughly 15-40%). The
median CTR among visible pages is 0.21%, and even the 75th percentile
(0.43%) sits below the 0.5% cutoff — meaning three out of four visible
pages get flagged. A rule meant to prioritize a manageable review subset
is instead flagging most of the eligible catalog, which limits its
practical usefulness for triage (this is exactly the coverage problem
the capstone's model was built to help address by ranking within the
flagged set, rather than treating every flagged page as equal priority).

**Verdict: the flag's underlying assumption doesn't hold as currently
calibrated.** A benchmark closer to the median (~0.2%) or the 25th
percentile (~0.09%) would flag a more genuinely distinct minority of
pages, and would be worth testing as an alternative in a future revision
of the baseline rule — not changed here, since altering an already-shipped
Week 4 deliverable is out of scope for this audit, but the finding is
worth surfacing rather than leaving unexamined.

In [6]:
# Check where 0.5% CTR actually falls in the distribution of visible pages
# (position <= 20, meeting the volume floor) — is it a reasonable threshold,
# or is it flagging almost everyone / almost no one?

visible = eligible[(eligible["avg_position"] > 0) & (eligible["avg_position"] <= 20)]
print(f"Visible, eligible pages (position<=20, impressions>=500): {len(visible)}")
print()
print("CTR percentile distribution among visible pages:")
print(visible["ctr"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_string())
print()

pct_below_benchmark = (visible["ctr"] < 0.5).mean()
print(f"Share of visible pages below the 0.5% CTR benchmark: {round(100*pct_below_benchmark, 1)}%")
print()
print("A benchmark that flags ~5% or ~50% of visible pages would be suspect")
print("(too rare to matter, or too common to be a meaningful signal).")
print("A benchmark flagging a genuine minority (roughly 15-40%) is a reasonable, defensible cut point.")


Visible, eligible pages (position<=20, impressions>=500): 50764

CTR percentile distribution among visible pages:
count    50764.000000
mean         0.315136
std          0.369887
min          0.000000
10%          0.000000
25%          0.090000
50%          0.210000
75%          0.430000
90%          0.720000
max          9.010000

Share of visible pages below the 0.5% CTR benchmark: 80.8%

A benchmark that flags ~5% or ~50% of visible pages would be suspect
(too rare to matter, or too common to be a meaningful signal).
A benchmark flagging a genuine minority (roughly 15-40%) is a reasonable, defensible cut point.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Position and CTR move together as expected (Signal 1, cleanly confirmed),
and CTR itself is entangled with impression volume (Signal 3) — meaning
any CTR-based rule should account for volume, not treat CTR in isolation.
The impression-vs-position relationship isn't as clean as assumed
(Signal 2, mixed), so a rule that infers "low position implies low
volume" will sometimes be wrong. Most importantly: the existing
`low_ctr_visible_page` flag's 0.5% threshold catches 80.8% of visible
pages rather than a distinct minority, which means the flag alone doesn't
do enough to separate "needs review" from "normal" — the ranked model in
the capstone project exists specifically to add that missing
prioritization layer on top of an overly broad flag.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.